# NER Text Masker: evaluation on public datasets
**English:** Kaggle *Name Entity Recognition (NER) Dataset* (`debasisdotcom/name-entity-recognition-ner-dataset`, GMB annotations)
**Arabic:** ANERCorp test split, downloaded from GitHub (`CSabty/ANERcorp-Correction`), with two corrected versions of the same sentences (`CSabty` corrected labels, `iwan-rg/CLEANANERCorp`)

The previous notebook used 18 hand-written sentences. This one measures the same models on thousands of annotated sentences:

| Text | Models |
|---|---|
| English | `dslim/bert-base-NER` (pre-trained) vs `numind/NuNER_Zero` (zero-shot) |
| Arabic | `urchade/gliner_multi-v2.1`, `numind/NuNER_Zero`, `CAMeL-Lab/bert-base-arabic-camelbert-mix-ner`, and `dslim/bert-base-NER` as a baseline |

For every model: exact **Precision / Recall / F1**, per-label F1, **masking recall** (share of entities whose every character is hidden), a paired bootstrap confidence interval for the F1 difference, the zero-shot threshold curve, speed on GPU and CPU, and the most frequent missed entities.

**How to run**
* **Colab:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. The English dataset downloads with `kagglehub`, the Arabic files from GitHub; nothing has to be uploaded.
* **Kaggle:** add both datasets with *Add Input* and turn on the T4 GPU. The paths under `/kaggle/input/` are found automatically.
* If a download asks for credentials, add `KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab secrets (key icon on the left).

## 0. Setup

In [ ]:
!pip install -q gliner kagglehub

import importlib.metadata as md, torch
for pkg in ["torch", "transformers", "gliner", "kagglehub"]:
    print(f"{pkg:13s}", md.version(pkg))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only)")

The model helpers from the comparison notebook (loading, chunking, overlap resolution, scores).

In [ ]:
%%writefile anonymizer_core.py
"""
anonymizer_core.py - building blocks for the NER Text Masker assignment.

Pipeline:  read file -> detect entities (NER + regex) -> resolve overlaps
           -> build ONE consistent replacement per entity -> replace by offsets
           -> rebuild the .txt / .docx

Models are imported lazily, so everything except detect_bert / detect_zero_shot
can be unit-tested without downloading a model.
"""
import re
import time
import warnings

from faker import Faker

BERT_MODEL = "dslim/bert-base-NER"
ZERO_SHOT_MODEL = "numind/NuNER_Zero"

# ---------------------------------------------------------------- labels
# Every model names its labels differently; the app works with one set.
LABEL_MAP = {
    "PER": "PER", "PERS": "PER", "PERSON": "PER",
    "ORG": "ORG", "ORGANIZATION": "ORG", "ORGANISATION": "ORG", "COMPANY": "ORG",
    "LOC": "LOC", "LOCATION": "LOC", "GPE": "LOC", "CITY": "LOC", "COUNTRY": "LOC",
}
# A zero-shot model reads the label as text, so it gets words, not codes.
ZS_PROMPT = {"PER": "person", "ORG": "organization", "LOC": "location"}


def norm_label(label: str) -> str:
    key = label.strip().upper()
    return LABEL_MAP.get(key, key.replace(" ", "_"))


# ---------------------------------------------------------------- anonymizer
class Anonymizer:
    """Keeps ONE replacement per entity, so the same name is replaced the same way
    everywhere in the document, and short forms ("Ahmed") follow the full name."""

    METHODS = ("mask", "tag", "fake", "partial", "generalize")

    def __init__(self, method="mask", locale="en_US", seed=42):
        if method not in self.METHODS:
            raise ValueError(f"method must be one of {self.METHODS}")
        self.method = method
        self.fake = Faker(locale)
        self.fake.seed_instance(seed)          # reproducible output for the report
        self.mapping, self.counters = {}, {}

    def _make(self, text, label):
        n = self.counters[label] = self.counters.get(label, 0) + 1
        fakers = {"PER": self.fake.name, "ORG": self.fake.company, "LOC": self.fake.city,
                  "EMAIL": self.fake.email, "PHONE": self.fake.phone_number}
        return {
            "mask":       lambda: re.sub(r"\S", "*", text),          # keeps spaces and length
            "tag":        lambda: f"[{label}_{n}]",
            "fake":       lambda: fakers.get(label, lambda: f"[{label}]")(),
            "partial":    lambda: " ".join(w[0] + "*" * (len(w) - 1) for w in text.split()),
            "generalize": lambda: {"PER": "[PERSON]", "ORG": "[COMPANY]",
                                   "LOC": "[PLACE]"}.get(label, f"[{label}]"),
        }[self.method]()

    def _alias(self, key):
        """'Ahmed' after 'Ahmed Ali' -> reuse the matching part of the same replacement."""
        label, name = key
        if label != "PER" or " " in name:
            return None
        for (lab, full), rep in self.mapping.items():
            parts = full.split()
            if lab == "PER" and len(parts) > 1 and name in parts:
                rp = rep.split()
                return rp[min(parts.index(name), len(rp) - 1)]
        return None

    def prepare(self, entities):
        """Pass 1: register every (text, label), longest first, so full names
        get their replacement before their short forms."""
        for text, label in sorted(entities, key=lambda x: -len(x[0])):
            self.replace(text, label)

    def replace(self, text, label):
        key = (label, " ".join(text.lower().split()))
        if key not in self.mapping:
            self.mapping[key] = self._alias(key) or self._make(text, label)
        return self.mapping[key]


# ---------------------------------------------------------------- spans
def resolve_overlaps(ents):
    """Drop duplicates (chunk overlap, regex + NER) and keep the longer span."""
    out = []
    for e in sorted(ents, key=lambda e: (e["start"], -(e["end"] - e["start"]))):
        if out and e["start"] < out[-1]["end"]:
            if e["end"] - e["start"] > out[-1]["end"] - out[-1]["start"]:
                out[-1] = e
            continue
        out.append(e)
    return out


def merge_adjacent(ents, text):
    """Zero-shot models may split one name into neighbouring spans of the same label."""
    out = []
    for e in sorted(ents, key=lambda e: e["start"]):
        if out and e["label"] == out[-1]["label"] and text[out[-1]["end"]:e["start"]] in ("", " "):
            out[-1] = {**out[-1], "end": e["end"], "score": min(out[-1]["score"], e["score"])}
        else:
            out.append(dict(e))
    return out


def split_at_newlines(ents, text):
    """An entity never crosses a line break: a span that does would swallow the
    newline on replacement and break the document's line structure."""
    out = []
    for e in ents:
        pos = e["start"]
        for part in re.split(r"(\r\n|\r|\n)", text[e["start"]:e["end"]]):
            if part and part not in ("\r\n", "\r", "\n") and part.strip():
                lead = len(part) - len(part.lstrip())
                out.append({**e, "start": pos + lead, "end": pos + len(part.rstrip())})
            pos += len(part)
    return out


def apply(text, ents, anon):
    """Replace from the END of the text so earlier offsets stay valid."""
    for e in sorted(ents, key=lambda e: e["start"], reverse=True):
        rep = anon.replace(text[e["start"]:e["end"]], e["label"])
        text = text[:e["start"]] + rep + text[e["end"]:]
    return text


def chunks(text, max_chars=1500, overlap=200):
    """Yield (offset, chunk) windows that never cut a word, with overlap so an
    entity on a boundary is seen whole in at least one window."""
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            cut = text.rfind(" ", start, end)
            end = cut if cut > start else end
        yield start, text[start:end]
        if end >= len(text):
            break
        nxt = text.find(" ", end - overlap, end)
        start = nxt + 1 if nxt != -1 and nxt + 1 > start else end


# ---------------------------------------------------------------- regex PII
PATTERNS = {
    "EMAIL": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    "PHONE": r"(?<!\w)\+?\d[\d\s().-]{7,}\d(?!\w)",
}


def regex_entities(text, wanted):
    out = []
    for lab, pat in PATTERNS.items():
        if lab not in wanted:
            continue
        for m in re.finditer(pat, text):
            if lab == "PHONE" and sum(c.isdigit() for c in m.group()) < 9:
                continue                      # skip dates like 2024-10-31
            out.append({"start": m.start(), "end": m.end(), "label": lab, "score": 1.0})
    return out


# ---------------------------------------------------------------- NER models
_models = {}


def load_bert(model_name=BERT_MODEL, device="cpu"):
    key = (model_name, device)
    if key not in _models:
        from transformers import pipeline
        ner = pipeline("token-classification", model=model_name,
                       aggregation_strategy="first",            # whole words, not ##subwords
                       device=0 if str(device).startswith("cuda") else -1)
        # Checkpoints without model_max_length get a huge default; the pipeline then
        # never splits long inputs and BERT crashes past 512 tokens.
        if ner.tokenizer.model_max_length > 100_000:
            ner.tokenizer.model_max_length = ner.model.config.max_position_embeddings
        _models[key] = ner
    return _models[key]


def detect_bert(text, model_name=BERT_MODEL, device="cpu", stride=128):
    """Pre-trained NER. stride= makes the pipeline cover texts longer than 512
    tokens with overlapping windows; manual chunking is the fallback."""
    ner = load_bert(model_name, device)
    try:
        raw = [(0, e) for e in ner(text, stride=stride)]
    except Exception as ex:                                     # slow tokenizer / old version
        warnings.warn(f"stride not supported ({ex}); chunking manually")
        raw = [(off, e) for off, chunk in chunks(text) for e in ner(chunk)]
    return resolve_overlaps([{"start": e["start"] + off, "end": e["end"] + off,
                              "label": norm_label(e["entity_group"]), "score": float(e["score"])}
                             for off, e in raw])


def load_zero_shot(model_name=ZERO_SHOT_MODEL, device="cpu"):
    key = (model_name, device)
    if key not in _models:
        from gliner import GLiNER
        model = GLiNER.from_pretrained(model_name, map_location=device)
        model.to(device)
        model.eval()
        _models[key] = model
    return _models[key]


def detect_zero_shot(text, labels=("PER", "ORG", "LOC"), threshold=0.5,
                     model_name=ZERO_SHOT_MODEL, device="cpu", merge=True):
    """Zero-shot NER (GLiNER family). Labels are free text chosen by the user;
    PER/ORG/LOC are sent as words and NuNER Zero expects lower case."""
    model = load_zero_shot(model_name, device)
    prompts = [ZS_PROMPT.get(l.upper(), l).lower() for l in labels]
    ents = []
    for off, chunk in chunks(text):                 # GLiNER truncates long inputs otherwise
        for e in model.predict_entities(chunk, prompts, threshold=threshold):
            ents.append({"start": e["start"] + off, "end": e["end"] + off,
                         "label": norm_label(e["label"]), "score": float(e["score"])})
    ents = resolve_overlaps(ents)
    return merge_adjacent(ents, text) if merge else ents


def detect(text, model="bert", labels=("PER", "ORG", "LOC"), threshold=0.5,
           regex_labels=("EMAIL", "PHONE"), device="cpu"):
    wanted = {norm_label(l) for l in labels}
    if model == "bert":
        ents = [e for e in detect_bert(text, device=device) if e["label"] in wanted]
    else:
        ents = detect_zero_shot(text, labels, threshold, device=device)
    ents += regex_entities(text, set(regex_labels))
    return resolve_overlaps(split_at_newlines(ents, text))


def count_params(model):
    """Millions of parameters of a pipeline or a GLiNER model."""
    module = getattr(model, "model", model)
    if not hasattr(module, "parameters"):
        module = model
    return sum(p.numel() for p in module.parameters()) / 1e6


# ---------------------------------------------------------------- .txt
def read_txt(path):
    """newline='' keeps \\r\\n exactly; cp1256 covers Arabic files saved by old Windows tools."""
    for enc in ("utf-8-sig", "cp1256", "latin-1"):
        try:
            with open(path, encoding=enc, newline="") as f:
                return f.read(), enc
        except UnicodeDecodeError:
            continue


def write_txt(path, text, encoding="utf-8"):
    with open(path, "w", encoding=encoding, newline="") as f:
        f.write(text)


def anonymize_txt(src, dst, anon, **detect_kw):
    text, enc = read_txt(src)
    ents = detect(text, **detect_kw)
    anon.prepare([(text[e["start"]:e["end"]], e["label"]) for e in ents])
    write_txt(dst, apply(text, ents, anon), "utf-8" if enc == "latin-1" else enc.replace("-sig", ""))
    return text, ents


# ---------------------------------------------------------------- .docx
def all_runs(par):
    """paragraph.runs skips text inside hyperlinks - include those runs too."""
    from docx.text.hyperlink import Hyperlink
    for item in par.iter_inner_content():
        yield from (item.runs if isinstance(item, Hyperlink) else [item])


def iter_paragraphs(doc):
    """Body, tables (nested; a merged cell only once), headers and footers."""
    seen = set()

    def walk(container):
        for p in container.paragraphs:
            if p._p not in seen:
                seen.add(p._p)
                yield p
        for t in container.tables:
            for row in t.rows:
                for cell in row.cells:
                    yield from walk(cell)

    yield from walk(doc)
    for s in doc.sections:
        for hf in (s.header, s.footer):
            if not hf.is_linked_to_previous:
                yield from walk(hf)


def replace_in_runs(runs, start, end, rep):
    """Replace paragraph-level [start, end) even when Word split it over several runs.
    The replacement takes the formatting of the run it starts in."""
    pos, first = 0, True
    for r in runs:
        t = r.text
        s, e = pos, pos + len(t)
        pos = e
        if e <= start or s >= end:
            continue
        a, b = max(start, s) - s, min(end, e) - s
        r.text = t[:a] + (rep if first else "") + t[b:]
        first = False


def anonymize_docx(src, dst, anon, **detect_kw):
    from docx import Document
    doc = Document(src)
    found = []                                            # pass 1: detect everything
    for par in iter_paragraphs(doc):
        runs = list(all_runs(par))
        text = "".join(r.text for r in runs)
        found.append((runs, text, detect(text, **detect_kw) if text.strip() else []))
    anon.prepare([(t[e["start"]:e["end"]], e["label"]) for _, t, ents in found for e in ents])
    for runs, text, ents in found:                        # pass 2: replace
        for e in sorted(ents, key=lambda e: e["start"], reverse=True):
            replace_in_runs(runs, e["start"], e["end"],
                            anon.replace(text[e["start"]:e["end"]], e["label"]))
    doc.save(dst)
    return found


# ---------------------------------------------------------------- evaluation
def build_gold(raw):
    """raw = [(sentence, [(entity_text, label[, occurrence]), ...]), ...] -> texts, gold spans."""
    texts, gold = [], []
    for sent, ents in raw:
        spans = []
        for item in ents:
            ent, lab, occ = (item + (0,))[:3]
            i = -1
            for _ in range(occ + 1):
                i = sent.find(ent, i + 1)
            if i < 0:
                raise ValueError(f"{ent!r} not found in {sent!r}")
            spans.append({"start": i, "end": i + len(ent), "label": lab})
        texts.append(sent)
        gold.append(sorted(spans, key=lambda e: e["start"]))
    return texts, gold


def _keep(ents, labels):
    return {(e["start"], e["end"], e["label"]) for e in ents if labels is None or e["label"] in labels}


def exact_scores(preds, golds, labels=None):
    """Strict NER metric: span AND label must match exactly (micro-averaged)."""
    tp = fp = fn = 0
    for p, g in zip(preds, golds):
        P, G = _keep(p, labels), _keep(g, labels)
        tp, fp, fn = tp + len(P & G), fp + len(P - G), fn + len(G - P)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"Precision": round(prec, 3), "Recall": round(rec, 3), "F1": round(f1, 3),
            "TP": tp, "FP": fp, "FN": fn}


def masking_scores(preds, golds, texts):
    """Anonymization view: is every character of the real entity hidden, whatever
    label the model gave it? A wrong label still hides a name; a missed word leaks it."""
    full = part = leak = 0
    for p, g, t in zip(preds, golds, texts):
        for e in g:
            chars = [i for i in range(e["start"], e["end"]) if not t[i].isspace()]
            hit = sum(any(x["start"] <= i < x["end"] for x in p) for i in chars)
            full += hit == len(chars)
            part += 0 < hit < len(chars)
            leak += hit == 0
    total = full + part + leak
    return {"Masking recall": round(full / total, 3) if total else 0.0,
            "Partly leaked": part, "Fully leaked": leak}


def benchmark(fn, texts, warmup=1):
    """Seconds per 1000 words; warm-up calls (model load, CUDA init) are not timed."""
    for t in texts[:warmup]:
        fn(t)
    try:
        import torch
        sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    except ImportError:
        sync = lambda: None
    words = sum(len(t.split()) for t in texts)
    sync()
    t0 = time.perf_counter()
    for t in texts:
        fn(t)
    sync()
    return (time.perf_counter() - t0) / words * 1000

### Configuration
`N_EN` and `N_AR` are the number of sentences sampled from each dataset (fixed seed, so runs are comparable). Set them to `None` to use every sentence; on a T4 the full English set (~48k sentences) takes roughly 15 minutes for NuNER Zero.

In [ ]:
# ---- datasets
EN_KAGGLE_HANDLE = "debasisdotcom/name-entity-recognition-ner-dataset"
# Arabic: ANERCorp test split from GitHub. "kaggle" uses the whole corpus from AR_PATH / AR_KAGGLE_HANDLE instead.
AR_SOURCES = {
    "anercorp_test":           "https://raw.githubusercontent.com/CSabty/ANERcorp-Correction/main/ANERcorp_test_original.txt",
    "anercorp_test_corrected": "https://raw.githubusercontent.com/CSabty/ANERcorp-Correction/main/ANERcorp_test_corrected.txt",
    "cleananercorp_test":      "https://raw.githubusercontent.com/iwan-rg/CLEANANERCorp/main/cleananercorp_TEST.txt",
}
AR_SOURCE = "anercorp_test"                            # main Arabic evaluation
AR_EXTRA_SOURCES = ["anercorp_test_corrected", "cleananercorp_test"]   # same sentences, corrected labels
AR_PATH = "/kaggle/input/arabic-corpus/ANERCorp.txt"   # only for AR_SOURCE = "kaggle"
AR_KAGGLE_HANDLE = ""                                  # only for AR_SOURCE = "kaggle", e.g. "owner/arabic-corpus"
N_EN, N_AR, SEED = 2000, None, 42                      # None = every sentence (the Arabic test split has ~830)
MAX_TOKENS = 80          # longer Arabic "sentences" are cut into pieces of this many tokens

# ---- models
EN_MODELS = {
    "dslim BERT":  {"kind": "bert",   "id": "dslim/bert-base-NER"},
    "NuNER Zero":  {"kind": "gliner", "id": "numind/NuNER_Zero", "threshold": 0.7, "merge": True},
}
AR_MODELS = {
    "gliner_multi": {"kind": "gliner", "id": "urchade/gliner_multi-v2.1", "threshold": 0.4, "merge": False,
                     "distractors": ["nationality", "date"]},
    "NuNER Zero":   {"kind": "gliner", "id": "numind/NuNER_Zero", "threshold": 0.7, "merge": True},
    "CAMeLBERT":    {"kind": "bert",   "id": "CAMeL-Lab/bert-base-arabic-camelbert-mix-ner"},
    "dslim BERT":   {"kind": "bert",   "id": "dslim/bert-base-NER"},
}
CORE = ("PER", "ORG", "LOC")
PROMPTS = {"PER": "person", "ORG": "organization", "LOC": "location", "DATE": "date", "NAT": "nationality"}
SPEED_N = 150            # sentences per model for the CPU / GPU speed test

In [ ]:
import collections, glob, importlib, json, os, random, re, time, warnings
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import anonymizer_core as ac
importlib.reload(ac)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("Main device:", DEV)

# Chart style (validated categorical colours; text stays in ink colours)
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#8a5cd6"]

def style(ax, title, ylabel):
    ax.set_facecolor(SURF); ax.figure.set_facecolor(SURF)
    ax.set_title(title, loc="left", color=INK, fontsize=12, pad=12)
    ax.set_ylabel(ylabel, color=INK2)
    ax.tick_params(colors=INK2, length=0)
    ax.grid(axis="y", color=GRID, lw=0.8); ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)

COUNTS = ["TP", "FP", "FN", "Partly leaked", "Fully leaked", "Entities"]

def scores_df(rows):
    df = pd.DataFrame(rows).T.astype(float)
    for c in COUNTS:
        if c in df:
            df[c] = df[c].astype(int)
    return df

def kaggle_credentials():
    # Colab secrets -> environment, so kagglehub can download datasets that need a login
    try:
        from google.colab import userdata
        for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            if not os.environ.get(k):
                os.environ[k] = userdata.get(k)
    except Exception:
        pass

## 1. English dataset (GMB annotations)
One row per token: `Sentence #`, `Word`, `POS`, `Tag` (BIO tags). Entity types are mapped to the app's labels:

| GMB | Here | Evaluated with |
|---|---|---|
| `per` | PER | both models |
| `org` | ORG | both models |
| `geo` | LOC | both models |
| `tim` | DATE | NuNER Zero only (dslim has no such label) |
| `gpe` | NAT (nationality / political group) | NuNER Zero only |
| `art`, `eve`, `nat` | not used | |

Tokens are joined with single spaces, so gold offsets are exact.

In [ ]:
def find_files(pattern):
    return sorted(glob.glob(pattern, recursive=True))

def en_dataset_csv():
    hits = [p for p in find_files("/kaggle/input/**/*.csv")]
    if not hits:
        kaggle_credentials()
        import kagglehub
        root = kagglehub.dataset_download(EN_KAGGLE_HANDLE)
        hits = find_files(os.path.join(root, "**", "*.csv"))
    for path in hits:
        for enc in ("utf-8", "latin-1"):
            try:
                head = pd.read_csv(path, nrows=5, encoding=enc)
            except Exception:
                continue
            cols = {c.lower().strip(): c for c in head.columns}
            if any("word" in c for c in cols) and any(c == "tag" or c.endswith("tag") for c in cols):
                return path, enc
    raise FileNotFoundError(f"No token/tag CSV found in {hits}")

EN_CSV, EN_ENC = en_dataset_csv()
raw = pd.read_csv(EN_CSV, encoding=EN_ENC)
cols = {c.lower().strip(): c for c in raw.columns}
SENT_COL = next(cols[c] for c in cols if "sentence" in c)
WORD_COL = next(cols[c] for c in cols if "word" in c)
TAG_COL = next(cols[c] for c in cols if c == "tag" or c.endswith("tag"))
raw[SENT_COL] = raw[SENT_COL].ffill()
raw[WORD_COL] = raw[WORD_COL].astype(str)
print(EN_CSV, f"({EN_ENC})", raw.shape)
print(raw[TAG_COL].value_counts().head(17).to_string())

In [ ]:
EN_MAP = {"per": "PER", "org": "ORG", "geo": "LOC", "tim": "DATE", "gpe": "NAT"}
AR_MAP = {"pers": "PER", "per": "PER", "org": "ORG", "loc": "LOC"}

def bio_to_example(tokens, tags, type_map):
    # join tokens with spaces and turn BIO tags into character spans
    text, spans, pos, cur = "", [], 0, None
    for tok, tag in zip(tokens, tags):
        start = pos if not text else pos + 1
        text = tok if not text else text + " " + tok
        pos = start + len(tok)
        tag = str(tag).strip()
        prefix, _, typ = tag.partition("-")
        label = type_map.get(typ.lower()) if typ else None
        if prefix == "I" and cur is not None and cur["label"] == label:
            cur["end"] = pos
            continue
        cur = None
        if prefix in ("B", "I") and label:
            cur = {"start": start, "end": pos, "label": label}
            spans.append(cur)
    return text, spans

EN_ALL = [bio_to_example(g[WORD_COL].tolist(), g[TAG_COL].tolist(), EN_MAP)
          for _, g in raw.groupby(SENT_COL, sort=False)]
rng = random.Random(SEED)
EN = EN_ALL if N_EN is None else rng.sample(EN_ALL, min(N_EN, len(EN_ALL)))
EN_TEXTS = [t for t, _ in EN]
EN_GOLD = [g for _, g in EN]
cnt = collections.Counter(e["label"] for g in EN_GOLD for e in g)
print(f"{len(EN_ALL)} sentences in the dataset, {len(EN)} sampled, {sum(len(t.split()) for t in EN_TEXTS)} words")
print("Gold entities in the sample:", dict(cnt))
t, g = EN[0]
print("\nExample:", t)
print([f'{t[e["start"]:e["end"]]}/{e["label"]}' for e in g])

## 2. Arabic dataset (ANERCorp test split from GitHub)
ANERCorp (Benajiba et al., 2007) is the standard Arabic NER benchmark: one token per line with its tag (`B-PERS`, `I-LOC`, `B-ORG`, `B-MISC`, `O`) and a blank line between sentences. Three versions of the **test split** are downloaded from public GitHub repositories:

| Source | Repository | What it is |
|---|---|---|
| `anercorp_test` | `CSabty/ANERcorp-Correction` | the original labels (main evaluation) |
| `anercorp_test_corrected` | `CSabty/ANERcorp-Correction` | the same sentences with obvious label errors fixed (Hatab et al., LREC 2022) |
| `cleananercorp_test` | `iwan-rg/CLEANANERCorp` | another corrected version (AlDuwais et al., OSACT 2024) |

The corrected versions show how much of each model's "error" is really a label error in the benchmark. MISC is not evaluated; extra columns in a line (such as `REVISIT`) are ignored.

> **Note on CAMeLBERT:** this model was fine-tuned on ANERCorp. Only the test split is used here, so its score is fair if its training used the same split; otherwise it is optimistic.

In [ ]:
import urllib.request

def download(url, folder="data"):
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, os.path.basename(url))
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    return path

def ar_dataset_path(source):
    if source in AR_SOURCES:
        return download(AR_SOURCES[source])
    # source == "kaggle": the whole corpus from a Kaggle input folder or dataset
    if os.path.exists(AR_PATH):
        return AR_PATH
    hits = find_files("/kaggle/input/**/ANERCorp*")
    if hits:
        return hits[0]
    if AR_KAGGLE_HANDLE:
        kaggle_credentials()
        import kagglehub
        root = kagglehub.dataset_download(AR_KAGGLE_HANDLE)
        hits = [p for p in find_files(os.path.join(root, "**", "*")) if "anercorp" in os.path.basename(p).lower()]
        if hits:
            return hits[0]
    raise FileNotFoundError("ANERCorp not found: use a GitHub source, or set AR_PATH / AR_KAGGLE_HANDLE")

BIO = re.compile(r"^(O|[BI]-[A-Za-z]+)$")

def read_anercorp(path):
    for enc in ("utf-8-sig", "cp1256"):
        try:
            lines = open(path, encoding=enc).read().splitlines()
            break
        except UnicodeDecodeError:
            continue
    has_blank = any(not l.strip() for l in lines)
    sents, toks, tags = [], [], []
    def flush():
        for i in range(0, len(toks), MAX_TOKENS):
            sents.append((toks[i:i + MAX_TOKENS], tags[i:i + MAX_TOKENS]))
        toks.clear(); tags.clear()
    for line in lines:
        parts = line.split()
        if not parts:
            flush(); continue
        tag = next((p for p in parts[1:] if BIO.match(p)), None)
        if tag is None:
            continue
        toks.append(parts[0]); tags.append(tag)
        if not has_blank and parts[0] in (".", "؟", "!", "?"):   # whole corpus without sentence breaks
            flush()
    flush()
    return sents

def load_arabic(source, sample=N_AR):
    path = ar_dataset_path(source)
    sents = read_anercorp(path)
    examples = [x for x in (bio_to_example(t, g, AR_MAP) for t, g in sents) if x[0].strip()]
    if sample is not None and sample < len(examples):
        examples = random.Random(SEED).sample(examples, sample)
    return path, sents, examples

AR_FILE, AR_SENTS, AR = load_arabic(AR_SOURCE)
AR_TEXTS = [t for t, _ in AR]
AR_GOLD = [g for _, g in AR]
tag_counts = collections.Counter(tag for _, tags in AR_SENTS for tag in tags)
print(AR_FILE, f"{sum(len(t) for t, _ in AR_SENTS)} tokens, {len(AR)} sentences used")
print(dict(tag_counts.most_common(10)))
print("Gold entities:", dict(collections.Counter(e["label"] for g in AR_GOLD for e in g)))
t, g = AR[0]
print("\nExample:", t)
print([f'{t[e["start"]:e["end"]]}/{e["label"]}' for e in g])

## 3. Models and prediction
Sentences are sent in batches. Zero-shot models read the label as text (`person`, `organization`, ...); for `gliner_multi` two extra *distractor* labels (`nationality`, `date`) are asked too and their spans are dropped, which stops the model from forcing every capitalised or foreign word into PER/ORG/LOC. NuNER Zero is token-level, so neighbouring pieces with the same label are merged (*New* + *York*).

In [ ]:
LOADED, LOAD_INFO = {}, {}

def get_model(spec, device=DEV):
    key = (spec["id"], device)
    if key not in LOADED:
        t0 = time.perf_counter()
        model = ac.load_bert(spec["id"], device) if spec["kind"] == "bert" else ac.load_zero_shot(spec["id"], device)
        LOADED[key] = model
        if device == DEV:
            LOAD_INFO[spec["id"]] = {"Load time (s)": round(time.perf_counter() - t0, 1),
                                     "Parameters (M)": round(ac.count_params(model), 1)}
    return LOADED[key]

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def predict(spec, texts, labels, device=DEV, threshold=None, batch_size=32):
    # returns (predictions, seconds); predictions keep only the requested labels
    model = get_model(spec, device)
    keep = set(labels)
    sync(); t0 = time.perf_counter()
    if spec["kind"] == "bert":
        raw = model(list(texts), batch_size=batch_size)
        preds = [[{"start": int(e["start"]), "end": int(e["end"]), "label": ac.norm_label(e["entity_group"]),
                   "score": float(e["score"])} for e in out] for out in raw]
    else:
        prompts = [PROMPTS[l] for l in labels]
        prompts += [d for d in spec.get("distractors", []) if d not in prompts]
        back = {PROMPTS[l]: l for l in labels}
        thr = spec.get("threshold", 0.5) if threshold is None else threshold
        try:
            raw = model.inference(list(texts), prompts, threshold=thr, batch_size=batch_size)
        except (AttributeError, TypeError):
            raw = [model.predict_entities(t, prompts, threshold=thr) for t in texts]
        preds = [[{"start": int(e["start"]), "end": int(e["end"]), "label": back.get(e["label"], "_drop"),
                   "score": float(e["score"])} for e in out] for out in raw]
        if spec.get("merge"):
            preds = [ac.merge_adjacent(p, t) for p, t in zip(preds, texts)]
    sync(); secs = time.perf_counter() - t0
    preds = [ac.resolve_overlaps([e for e in p if e["label"] in keep]) for p in preds]
    return preds, secs

PROCLITIC = re.compile(r"^[وفبلك]{1,2}")

def lenient_masking(preds, golds, texts):
    # Arabic: ANERCorp tags whole tokens, so "والقاهرة" is gold LOC including "و".
    # A model that hides "القاهرة" but not the attached "و" still hides the name.
    full = 0
    for p, g, t in zip(preds, golds, texts):
        for e in g:
            s = e["start"]
            m = PROCLITIC.match(t[s:e["end"]])
            if m and e["end"] - s - m.end() >= 2:
                s += m.end()
            chars = [i for i in range(s, e["end"]) if not t[i].isspace()]
            full += all(any(x["start"] <= i < x["end"] for x in p) for i in chars)
    total = sum(len(g) for g in golds)
    return round(full / total, 3) if total else 0.0

def evaluate(preds, golds, texts, labels, arabic=False):
    golds = [[e for e in g if e["label"] in labels] for g in golds]
    out = {**ac.exact_scores(preds, golds), **ac.masking_scores(preds, golds, texts)}
    if arabic:
        out["Masking recall (proclitic-tolerant)"] = lenient_masking(preds, golds, texts)
    out["Entities"] = sum(len(g) for g in golds)
    return out

def per_label(preds, golds, labels):
    return {f"F1 {l}": ac.exact_scores(preds, [[e for e in g if e["label"] == l] for g in golds], labels={l})["F1"]
            for l in labels}

def per_sentence_counts(preds, golds, labels):
    rows = []
    for p, g in zip(preds, golds):
        P = {(e["start"], e["end"], e["label"]) for e in p if e["label"] in labels}
        G = {(e["start"], e["end"], e["label"]) for e in g if e["label"] in labels}
        rows.append((len(P & G), len(P - G), len(G - P)))
    return np.array(rows)

def bootstrap_f1_diff(a, b, n=1000, seed=SEED):
    # paired bootstrap over sentences: F1(a) - F1(b) with a 95% interval
    rng = np.random.default_rng(seed)
    def f1(c):
        tp, fp, fn = c.sum(0)
        return 2 * tp / (2 * tp + fp + fn) if tp else 0.0
    diffs = []
    for _ in range(n):
        idx = rng.integers(0, len(a), len(a))
        diffs.append(f1(a[idx]) - f1(b[idx]))
    return round(f1(a) - f1(b), 3), np.round(np.percentile(diffs, [2.5, 97.5]), 3)

## 4. English results: PER / ORG / LOC
Both models get the three labels the app sends to them.

In [ ]:
EN_PRED, EN_TIME = {}, {}
for name, spec in EN_MODELS.items():
    EN_PRED[name], EN_TIME[name] = predict(spec, EN_TEXTS, CORE)
    print(f"{name:12s} {EN_TIME[name]:7.1f} s")

EN_ACC = scores_df({n: {**evaluate(EN_PRED[n], EN_GOLD, EN_TEXTS, CORE), **per_label(EN_PRED[n], EN_GOLD, CORE)}
                    for n in EN_MODELS})
display(EN_ACC)

a, b = list(EN_MODELS)
diff, ci = bootstrap_f1_diff(per_sentence_counts(EN_PRED[a], EN_GOLD, CORE), per_sentence_counts(EN_PRED[b], EN_GOLD, CORE))
EN_DIFF = {"pair": f"{a} - {b}", "F1 difference": diff, "95% CI": ci.tolist()}
print(f"F1({a}) - F1({b}) = {diff:+.3f}, 95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]",
      "-> significant" if ci[0] > 0 or ci[1] < 0 else "-> not significant")

### Labels dslim cannot produce (DATE, NAT)
The zero-shot model is asked for `date` and `nationality`; dslim has no such labels, so its score is 0 by definition.

In [ ]:
EXTRA = ("DATE", "NAT")
zs_name = next(n for n, s in EN_MODELS.items() if s["kind"] == "gliner")
EXTRA_PRED, _ = predict(EN_MODELS[zs_name], EN_TEXTS, EXTRA)
EN_EXTRA = scores_df({zs_name: {**evaluate(EXTRA_PRED, EN_GOLD, EN_TEXTS, EXTRA), **per_label(EXTRA_PRED, EN_GOLD, EXTRA)}})
display(EN_EXTRA)

## 5. Arabic results: PER / ORG / LOC
*Masking recall (proclitic-tolerant)* counts an entity as hidden when everything except an attached `و ف ب ل ك` is covered: ANERCorp includes these letters in the gold token, while models (and the app) usually leave them out.

In [ ]:
AR_PRED, AR_TIME = {}, {}
for name, spec in AR_MODELS.items():
    try:
        AR_PRED[name], AR_TIME[name] = predict(spec, AR_TEXTS, CORE)
        print(f"{name:13s} {AR_TIME[name]:7.1f} s")
    except Exception as ex:                     # a failed download should not stop the notebook
        print(f"{name:13s} skipped: {type(ex).__name__}: {str(ex)[:150]}")

AR_ACC = scores_df({n: {**evaluate(AR_PRED[n], AR_GOLD, AR_TEXTS, CORE, arabic=True), **per_label(AR_PRED[n], AR_GOLD, CORE)}
                    for n in AR_PRED})
display(AR_ACC)

AR_DIFFS = []
names = list(AR_PRED)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        if "dslim BERT" in (names[i], names[j]):
            continue
        d, c = bootstrap_f1_diff(per_sentence_counts(AR_PRED[names[i]], AR_GOLD, CORE),
                                 per_sentence_counts(AR_PRED[names[j]], AR_GOLD, CORE))
        AR_DIFFS.append({"pair": f"{names[i]} - {names[j]}", "F1 difference": d, "95% CI": c.tolist(),
                         "significant": bool(c[0] > 0 or c[1] < 0)})
display(pd.DataFrame(AR_DIFFS))

### The same models on the corrected labels
Each corrected version is scored with the same models. A score that rises on the corrected labels means part of the error on the original test split was in the benchmark, not in the model.

In [ ]:
AR_VERSIONS = {AR_SOURCE: AR_ACC}
for src in AR_EXTRA_SOURCES:
    try:
        _, _, ex = load_arabic(src)
    except Exception as e:
        print(f"{src}: skipped ({type(e).__name__}: {str(e)[:120]})"); continue
    texts, golds = [t for t, _ in ex], [g for _, g in ex]
    rows = {}
    for name, spec in AR_MODELS.items():
        if name in AR_PRED:
            p, _ = predict(spec, texts, CORE)
            rows[name] = evaluate(p, golds, texts, CORE, arabic=True)
    AR_VERSIONS[src] = scores_df(rows)
    print(f"{src}: {len(texts)} sentences, {sum(len(g) for g in golds)} PER/ORG/LOC entities")

VERSIONS = pd.concat({src: df[["Precision", "Recall", "F1", "Masking recall (proclitic-tolerant)"]]
                      for src, df in AR_VERSIONS.items()}, axis=1)
display(VERSIONS)

## 6. Zero-shot threshold
Each zero-shot model predicts once at a low threshold (0.1); higher thresholds filter those scores.

In [ ]:
THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

def sweep(spec, texts, golds, arabic=False):
    low, _ = predict(spec, texts, CORE, threshold=0.1)
    rows = {}
    for th in THRESHOLDS:
        p = [[e for e in pp if e["score"] >= th] for pp in low]
        r = evaluate(p, golds, texts, CORE, arabic)
        rows[th] = {k: r[k] for k in ("Precision", "Recall", "F1", "Masking recall")}
    return pd.DataFrame(rows).T.rename_axis("threshold")

SWEEPS = {f"EN {zs_name}": sweep(EN_MODELS[zs_name], EN_TEXTS, EN_GOLD)}
if "gliner_multi" in AR_PRED:
    SWEEPS["AR gliner_multi"] = sweep(AR_MODELS["gliner_multi"], AR_TEXTS, AR_GOLD, arabic=True)
for name, df in SWEEPS.items():
    best = df["F1"].idxmax()
    print(f"{name}: best F1 {df.loc[best, 'F1']:.3f} at threshold {best}")
    display(df)

fig, axes = plt.subplots(1, len(SWEEPS), figsize=(6.5 * len(SWEEPS), 3.8), squeeze=False)
for ax, (name, df) in zip(axes[0], SWEEPS.items()):
    for col, color in zip(["Precision", "Recall", "F1", "Masking recall"], SERIES):
        ax.plot(df.index, df[col], marker="o", ms=4, lw=2, color=color, label=col)
    style(ax, name, "score"); ax.set_xlabel("threshold", color=INK2); ax.set_ylim(0, 1.02)
    ax.legend(frameon=False, fontsize=9, labelcolor=INK2)
plt.tight_layout(); plt.show()

## 7. Speed and resources
GPU time comes from the accuracy runs above (whole sample, batched). On a GPU runtime, CPU is measured on `SPEED_N` sentences per model with a second copy of the model on the CPU. Load time on the first run includes the download.

In [ ]:
def words(texts):
    return sum(len(t.split()) for t in texts)

rows = {}
for lang, models, texts, timing in (("EN", EN_MODELS, EN_TEXTS, EN_TIME), ("AR", AR_MODELS, AR_TEXTS, AR_TIME)):
    for name, spec in models.items():
        if name not in timing:
            continue
        row = {"Parameters (M)": LOAD_INFO[spec["id"]]["Parameters (M)"],
               "Load time (s)": LOAD_INFO[spec["id"]]["Load time (s)"]}
        if DEV == "cuda":
            row["s / 1000 words GPU"] = round(timing[name] / words(texts) * 1000, 3)
            sub = texts[:SPEED_N]
            predict(spec, sub[:5], CORE, device="cpu")              # warm-up
            _, cpu_s = predict(spec, sub, CORE, device="cpu", batch_size=8)
            LOADED.pop((spec["id"], "cpu"), None)                   # free the CPU copy
            ac._models.pop((spec["id"], "cpu"), None)
            row["s / 1000 words CPU"] = round(cpu_s / words(sub) * 1000, 3)
            row["GPU speed-up"] = round(row["s / 1000 words CPU"] / row["s / 1000 words GPU"], 1)
        else:
            row["s / 1000 words CPU"] = round(timing[name] / words(texts) * 1000, 3)
        rows[f"{lang} · {name}"] = row
SPEED = pd.DataFrame(rows).T
display(SPEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      "| GPU memory peak:", f"{torch.cuda.max_memory_allocated() / 2**30:.2f} GB" if torch.cuda.is_available() else "-")

## 8. Error analysis
The entities each model misses most often (exact span and label), and a few full sentences with every miss.

In [ ]:
def misses(preds, golds, texts, labels):
    out = collections.Counter()
    for p, g, t in zip(preds, golds, texts):
        P = {(e["start"], e["end"], e["label"]) for e in p}
        for e in g:
            if e["label"] in labels and (e["start"], e["end"], e["label"]) not in P:
                out[f'{t[e["start"]:e["end"]]} / {e["label"]}'] += 1
    return out

def examples(preds, golds, texts, labels, k=4):
    rows = []
    for p, g, t in zip(preds, golds, texts):
        G = {(e["start"], e["end"], e["label"]) for e in g if e["label"] in labels}
        P = {(e["start"], e["end"], e["label"]) for e in p}
        if G != P:
            rows.append({"sentence": t[:140],
                         "missed": [f"{t[s:e]}/{l}" for s, e, l in sorted(G - P)],
                         "extra / wrong": [f"{t[s:e]}/{l}" for s, e, l in sorted(P - G)]})
        if len(rows) == k:
            break
    return pd.DataFrame(rows)

for lang, preds, golds, texts in (("English", EN_PRED, EN_GOLD, EN_TEXTS), ("Arabic", AR_PRED, AR_GOLD, AR_TEXTS)):
    for name, p in preds.items():
        print(f"=== {lang} · {name}: most frequent misses")
        print(", ".join(f"{k} ×{v}" for k, v in misses(p, golds, texts, CORE).most_common(12)))
        display(examples(p, golds, texts, CORE))

## 9. Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (title, acc) in zip(axes, (("English (GMB)", EN_ACC), (f"Arabic ({AR_SOURCE})", AR_ACC))):
    x = np.arange(len(acc)); w = 0.38
    b1 = ax.bar(x - w / 2 - 0.01, acc["F1"], w, color=SERIES[0], label="F1 (exact)")
    b2 = ax.bar(x + w / 2 + 0.01, acc["Masking recall"], w, color=SERIES[1], label="Masking recall")
    for bars in (b1, b2):
        ax.bar_label(bars, fmt="%.2f", fontsize=8, color=INK2, padding=2)
    ax.set_xticks(x, acc.index); ax.set_ylim(0, 1.1)
    style(ax, title, "score"); ax.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="upper right")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (title, acc) in zip(axes, (("English: F1 per label", EN_ACC), ("Arabic: F1 per label", AR_ACC))):
    cols = [f"F1 {l}" for l in CORE]
    x = np.arange(len(acc)); w = 0.8 / len(cols)
    for i, (c, color) in enumerate(zip(cols, SERIES)):
        bars = ax.bar(x + (i - 1) * w, acc[c], w * 0.95, color=color, label=c.replace("F1 ", ""))
        ax.bar_label(bars, fmt="%.2f", fontsize=7, color=INK2, padding=1)
    ax.set_xticks(x, acc.index); ax.set_ylim(0, 1.1)
    style(ax, title, "F1"); ax.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="upper right")
plt.tight_layout(); plt.show()

## 10. Summary for the report
One table, plus observations generated from the numbers. Everything is saved to `kaggle_eval_results.json` and `kaggle_eval_summary.csv`.

In [ ]:
cols = ["Precision", "Recall", "F1", "Masking recall", "Partly leaked", "Fully leaked", "Entities"]
summary = pd.concat({"English (GMB)": EN_ACC[cols], f"Arabic ({AR_SOURCE})": AR_ACC[cols + ["Masking recall (proclitic-tolerant)"]]})
display(summary)

notes = []
best_en, best_ar = EN_ACC["F1"].idxmax(), AR_ACC["F1"].idxmax()
notes.append(f"English: best exact F1 {best_en} ({EN_ACC.loc[best_en, 'F1']:.3f}); "
             f"{EN_DIFF['pair']} = {EN_DIFF['F1 difference']:+.3f}, 95% CI {EN_DIFF['95% CI']}.")
for n in EN_ACC.index:
    notes.append(f"English {n}: masking recall {EN_ACC.loc[n, 'Masking recall']:.3f}, "
                 f"{EN_ACC.loc[n, 'Fully leaked']} of {EN_ACC.loc[n, 'Entities']} entities fully leaked.")
notes.append(f"English zero-shot extra labels: " + ", ".join(f"{c} {EN_EXTRA.loc[zs_name, c]:.3f}" for c in EN_EXTRA.columns if c.startswith("F1 ")) + ".")
notes.append(f"Arabic: best exact F1 {best_ar} ({AR_ACC.loc[best_ar, 'F1']:.3f}); CAMeLBERT was fine-tuned on ANERCorp (test split only is used here).")
for src, df in AR_VERSIONS.items():
    if src != AR_SOURCE:
        notes.append(f"Arabic on {src}: " + ", ".join(f"{n} F1 {df.loc[n, 'F1']:.3f}" for n in df.index) + ".")
for n in AR_ACC.index:
    notes.append(f"Arabic {n}: F1 {AR_ACC.loc[n, 'F1']:.3f}, masking recall {AR_ACC.loc[n, 'Masking recall']:.3f} "
                 f"(proclitic-tolerant {AR_ACC.loc[n, 'Masking recall (proclitic-tolerant)']:.3f}).")
for name, df in SWEEPS.items():
    notes.append(f"{name}: best F1 at threshold {df['F1'].idxmax()}.")
for idx, r in SPEED.iterrows():
    gpu = f"{r['s / 1000 words GPU']:.3f} s / 1000 words on GPU, " if "s / 1000 words GPU" in r else ""
    notes.append(f"{idx}: {gpu}{r['s / 1000 words CPU']:.3f} s / 1000 words on CPU.")
print("\n".join("- " + n for n in notes))

summary.to_csv("kaggle_eval_summary.csv")
with open("kaggle_eval_results.json", "w") as f:
    json.dump({"english": EN_ACC.reset_index().to_dict("records"), "english_extra": EN_EXTRA.reset_index().to_dict("records"),
               "arabic": AR_ACC.reset_index().to_dict("records"), "arabic_source": AR_SOURCE,
               "arabic_versions": {k: v.reset_index().to_dict("records") for k, v in AR_VERSIONS.items()}, "english_diff": EN_DIFF, "arabic_diffs": AR_DIFFS,
               "sweeps": {k: v.reset_index().to_dict("records") for k, v in SWEEPS.items()},
               "speed": SPEED.reset_index().to_dict("records"), "notes": notes,
               "sample": {"en_sentences": len(EN_TEXTS), "ar_sentences": len(AR_TEXTS), "seed": SEED, "ar_source": AR_SOURCE}},
              f, indent=2, ensure_ascii=False, default=str)
print("\nsaved kaggle_eval_summary.csv and kaggle_eval_results.json")

### Reading the results
* **Dataset conventions shape the exact scores.** GMB tags titles inside person names in some sentences and not in others, and ANERCorp includes attached letters (`و`, `ب`) in the entity. Boundary differences lower exact F1 without leaking anything, which is why masking recall is reported next to it.
* **Labels do not line up one to one.** GMB `geo` is close to LOC, but countries are sometimes `gpe`; dslim was trained on CoNLL, where nationalities are MISC. Some "errors" are label-scheme differences, not model mistakes.
* **CAMeLBERT and ANERCorp:** the model was fine-tuned on this corpus. Only the test split is used, but if its training split differs from this one, treat its numbers as an upper bound.
* **Label noise:** compare the original and corrected Arabic versions before blaming a model for an error.